# Predicting pressure drop in a pipe — the Python behind Explore

*Engineering ML Studio · Learn the Code · Notebook 1*

You have just used the browser **Explore** mode to predict the **pressure drop** of a fluid
flowing through a pipe. This notebook reproduces that *same* activity in Python, so you can see
and change the code behind it.

Same problem, same dataset, same fixed random seed (**42**), same three models, same plots and
the same engineering interpretation — just written out in code you can read, run and tweak.

> ⚠️ **This is a training demonstration, not an engineering design tool.** The dataset is
> **synthetic**, generated from a documented equation (Darcy–Weisbach) with a fixed seed and a
> little added noise. It is **not** experimental, validated or safety-grade data and **must not**
> be used for real design decisions. Machine learning here is a way to *learn*, not to replace
> engineering judgement or the underlying physics.

## 1. Learning objectives

By the end of this notebook you will be able to:

1. **Load** a small engineering dataset with `pandas` and inspect it.
2. **Choose inputs and a target** for a regression problem, keeping units in mind.
3. **Split** data reproducibly into training and test sets and explain *why* a fixed seed matters.
4. **Fit three models** — Linear Regression, a Decision Tree and a Random Forest — with
   `scikit-learn`.
5. **Measure** each model with **RMSE** and **R²** and read those numbers in physical units (kPa).
6. **See overfitting** by comparing training error with test error.
7. **Interpret the result like an engineer** — check physical trends, plausibility, and the
   difference between predicting *inside* and *outside* the data range (extrapolation).

No prior machine-learning experience is assumed. Model names are kept as secondary detail; the
engineering problem leads.

## 2. The engineering problem

A fluid flowing through a pipe loses pressure to **friction** along the pipe wall. Mechanical and
thermal engineers need to estimate this **pressure drop (Δp)** to size pumps, choose pipe
diameters and check whether a flow system will work at all.

We predict the pressure drop from a few properties of the pipe and the fluid:

| Input | Symbol | Unit |
| --- | --- | --- |
| Pipe length | L | m |
| Internal pipe diameter | D | m |
| Mean flow velocity | v | m/s |
| Fluid density | ρ | kg/m³ |
| Dynamic viscosity | μ | Pa·s |

**We predict:** pressure drop, **Δp**, in **kilopascals (kPa)**.

**Expected physical trends** (what an engineer already knows should happen):

- Pressure drop **increases** as **flow velocity** increases.
- Pressure drop **increases** as **pipe length** increases.
- Pressure drop **decreases** as **pipe diameter** increases.

The data was generated from the **Darcy–Weisbach** equation

$$\Delta p = f\,\frac{L}{D}\,\frac{\rho v^2}{2}$$

where the friction factor *f* comes from the Reynolds number $Re = \rho v D / \mu$ (all 500 rows in
this dataset are in **turbulent** flow). Wall roughness is held fixed, so it is *not* one of the
inputs. Because the data is synthetic there is a small amount of random scatter, so no model can be
perfect — which is realistic.

## 3. Load the dataset

First we import the libraries and load the bundled CSV. The loader tries a **local path first**
(so it works offline when you have the repository checked out) and only falls back to the
committed file on GitHub if no local copy is found (handy in Google Colab).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# One fixed seed, used everywhere, exactly as the browser Explore mode does.
SEED = 42
np.random.seed(SEED)

print("Library versions")
print("  numpy       ", np.__version__)
print("  pandas      ", pd.__version__)
import sklearn
print("  scikit-learn", sklearn.__version__)
print("Random seed:", SEED)

In [ ]:
CSV_NAME = "pipe_pressure_drop_sample.csv"
RAW_URL = (
    "https://raw.githubusercontent.com/weiwangstfc/engineering-ml-studio/"
    "main/examples/" + CSV_NAME
)

# Try local locations first (offline-friendly); fall back to the committed GitHub copy.
candidates = [
    os.path.join("..", "examples", CSV_NAME),  # running from notebooks/
    os.path.join("examples", CSV_NAME),         # running from the repo root
    CSV_NAME,                                     # running beside the CSV
]
csv_path = next((p for p in candidates if os.path.exists(p)), None)

if csv_path is not None:
    df = pd.read_csv(csv_path)
    print(f"Loaded local dataset: {csv_path}")
else:
    df = pd.read_csv(RAW_URL)  # e.g. in Colab
    print(f"Loaded dataset from GitHub: {RAW_URL}")

print("Rows, columns:", df.shape)
df.head()

## 4. Choose the inputs and the target

We name the five inputs and the single target explicitly. Keeping the units in the column names
(`_m`, `_m_s`, `_kpa`, …) is a small habit that prevents big mistakes.

In [ ]:
FEATURES = [
    "pipe_length_m",
    "pipe_diameter_m",
    "flow_velocity_m_s",
    "fluid_density_kg_m3",
    "dynamic_viscosity_pa_s",
]
TARGET = "pressure_drop_kpa"
TARGET_UNIT = "kPa"

X = df[FEATURES].to_numpy()
y = df[TARGET].to_numpy()

print("Inputs (features):")
for f in FEATURES:
    print("  -", f)
print("Target:", TARGET, f"({TARGET_UNIT})")
print("Number of rows:", len(y))

## 5. Inspect the data before modelling

Always *look* at the data first. The summary statistics show the range of each input and of the
pressure drop. The two scatter plots let us **see** two of the expected trends before any model is
fitted: Δp rising with velocity, and Δp falling as the pipe gets wider.

In [ ]:
df[FEATURES + [TARGET]].describe().T

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(df["flow_velocity_m_s"], df[TARGET], s=12, alpha=0.5, color="#125f76")
axes[0].set_xlabel("Flow velocity (m/s)")
axes[0].set_ylabel(f"Pressure drop ({TARGET_UNIT})")
axes[0].set_title("Δp tends to rise with velocity")

axes[1].scatter(df["pipe_diameter_m"], df[TARGET], s=12, alpha=0.5, color="#a52b2b")
axes[1].set_xlabel("Pipe diameter (m)")
axes[1].set_ylabel(f"Pressure drop ({TARGET_UNIT})")
axes[1].set_title("Δp tends to fall as diameter grows")

fig.tight_layout()
plt.show()

## 6. Split into training and test sets (reproducibly)

We hold back data the model never sees during training, so we can judge it honestly. Explore uses a
**70 % / 15 % / 15 %** split (training / validation / test) with a fixed seed. We reproduce the same
ratio and the same seed here.

**Why a fixed seed?** `random_state=42` makes the split repeatable: run the notebook again, or on
another machine, and you get the same partition — so results are comparable.

> **Note.** These three models are *not* tuned, so they do not use the validation set. It is created
> here only to mirror Explore's split; tuning on it is an advanced extension (Section 15). We train
> on the training set and report on the **test** set.

In [ ]:
# 70 / 15 / 15 : first hold out 30% (temp), then halve it into validation and test.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED
)

print(f"Training rows:   {len(y_train)}")
print(f"Validation rows: {len(y_val)}  (unused by these fixed models)")
print(f"Test rows:       {len(y_test)}")

## 7. The simple trend first — Linear Regression

**Linear Regression** learns a *weighted relationship*: each input is multiplied by a number and
the results are added up. It is easy to interpret, but because it combines the inputs in a
straight-line way it can struggle when the true relationship is curved — as pressure drop is (Δp
grows with roughly the square of velocity).

We put a `StandardScaler` in front of the linear model. Scaling puts the inputs on a comparable
footing; it does not change the quality of a plain linear fit, but it is good practice and matches
what Explore does internally.

In [ ]:
def rmse(y_true, y_pred):
    'Root-mean-square error, in the same units as the target (kPa).'
    return mean_squared_error(y_true, y_pred) ** 0.5

linear = make_pipeline(StandardScaler(), LinearRegression())
linear.fit(X_train, y_train)

lin_pred_train = linear.predict(X_train)
lin_pred_test = linear.predict(X_test)

print("Linear Regression")
print(f"  Train : R2 = {r2_score(y_train, lin_pred_train):.3f}   "
      f"RMSE = {rmse(y_train, lin_pred_train):6.2f} {TARGET_UNIT}")
print(f"  Test  : R2 = {r2_score(y_test, lin_pred_test):.3f}   "
      f"RMSE = {rmse(y_test, lin_pred_test):6.2f} {TARGET_UNIT}")

## 8. A more flexible relationship — Random Forest

A **Random Forest** combines many decision trees and averages them. It handles curved,
non-linear relationships well and is more stable than any single tree. The price is that it is
**less transparent** than a straight-line model, and it **extrapolates poorly** beyond the range of
the training data (more on that in Section 13).

We use scikit-learn's default forest settings with our fixed seed.

> **A note on settings.** Explore's in-browser forest uses slightly different settings (40 trees,
> `max_features='sqrt'`, depth 8). With only five inputs, `max_features='sqrt'` restricts each split
> to about two of them, and under scikit-learn the forest then does *worse* than a single tree —
> which would contradict what you saw in the browser. scikit-learn's defaults give the honest,
> expected ranking here, so we use them. Exact numbers will differ from the browser either way; the
> **story** is what matters (see Section 10).

In [ ]:
forest = RandomForestRegressor(random_state=SEED)  # scikit-learn defaults
forest.fit(X_train, y_train)

rf_pred_train = forest.predict(X_train)
rf_pred_test = forest.predict(X_test)

print("Random Forest")
print(f"  Train : R2 = {r2_score(y_train, rf_pred_train):.3f}   "
      f"RMSE = {rmse(y_train, rf_pred_train):6.2f} {TARGET_UNIT}")
print(f"  Test  : R2 = {r2_score(y_test, rf_pred_test):.3f}   "
      f"RMSE = {rmse(y_test, rf_pred_test):6.2f} {TARGET_UNIT}")

## 9. Seeing overfitting — a single Decision Tree

A **Decision Tree** splits the inputs into regions with a series of yes/no questions and predicts a
value for each region. It can capture non-linear behaviour, but a deep tree tends to **overfit**:
it fits the training rows almost perfectly yet does noticeably worse on unseen test rows.

Watch the gap between the training and test scores below — that gap *is* overfitting.

In [ ]:
tree = DecisionTreeRegressor(max_depth=8, min_samples_leaf=5, random_state=SEED)
tree.fit(X_train, y_train)

dt_pred_train = tree.predict(X_train)
dt_pred_test = tree.predict(X_test)

print("Decision Tree (max_depth=8, min_samples_leaf=5)")
print(f"  Train : R2 = {r2_score(y_train, dt_pred_train):.3f}   "
      f"RMSE = {rmse(y_train, dt_pred_train):6.2f} {TARGET_UNIT}")
print(f"  Test  : R2 = {r2_score(y_test, dt_pred_test):.3f}   "
      f"RMSE = {rmse(y_test, dt_pred_test):6.2f} {TARGET_UNIT}")
print()
gap = rmse(y_test, dt_pred_test) - rmse(y_train, dt_pred_train)
print(f"Test RMSE is {gap:.2f} {TARGET_UNIT} larger than training RMSE "
      f"-> a sign of overfitting.")

## 10. Compare the models — RMSE and R²

Now we put the three models side by side on the **test** set (the honest measure) and the training
set (to reveal overfitting).

- **R²** is the fraction of the variation in pressure drop the model explains: 1 is perfect, 0 is no
  better than always predicting the average.
- **RMSE** is the typical size of the prediction error, in **kPa** — smaller is better.

In [ ]:
def scores(name, model_pred_train, model_pred_test):
    return {
        "Model": name,
        "Train R2": round(r2_score(y_train, model_pred_train), 3),
        "Test R2": round(r2_score(y_test, model_pred_test), 3),
        f"Train RMSE ({TARGET_UNIT})": round(rmse(y_train, model_pred_train), 2),
        f"Test RMSE ({TARGET_UNIT})": round(rmse(y_test, model_pred_test), 2),
    }

results = pd.DataFrame([
    scores("Linear Regression", lin_pred_train, lin_pred_test),
    scores("Decision Tree", dt_pred_train, dt_pred_test),
    scores("Random Forest", rf_pred_train, rf_pred_test),
])
results

**Reading the table.** The linear model **under-fits** (lowest test R²) because the relationship is
curved. The single deep tree fits the training data very well but its **test** score is worse — the
classic overfitting gap. The **Random Forest** is the strongest and most stable on the test set.

This is the same qualitative ranking you saw in the browser (linear weakest → forest strongest). The
exact numbers differ from Explore because the browser uses its own JavaScript model code and a
different random partition of the rows — see the note in Section 12.

## 11. Actual vs predicted (the best model)

Each point is one **test** row: the true pressure drop on the horizontal axis, the model's
prediction on the vertical axis. Points on the dashed diagonal are perfect predictions; a tight
cloud along the line means accurate predictions.

In [ ]:
best_name, best_pred = "Random Forest", rf_pred_test

lo = float(min(y_test.min(), best_pred.min()))
hi = float(max(y_test.max(), best_pred.max()))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, best_pred, s=28, alpha=0.7, color="#125f76", label="Test rows")
ax.plot([lo, hi], [lo, hi], "--", color="#a52b2b", label="Perfect prediction")
ax.set_xlabel(f"Actual pressure drop ({TARGET_UNIT})")
ax.set_ylabel(f"Predicted pressure drop ({TARGET_UNIT})")
ax.set_title(f"{best_name}: actual vs predicted (test data)")
ax.legend()
fig.tight_layout()
plt.show()

## 12. Overfitting, and why the test score is the one that counts

A low **training** error on its own is never enough — a model can memorise training rows. What
matters is performance on data it has **not** seen. The chart below compares training and test RMSE
for all three models. A tall jump from training to test (as with the single tree) is overfitting.

In [ ]:
names = ["Linear Regression", "Decision Tree", "Random Forest"]
train_rmse = [rmse(y_train, lin_pred_train), rmse(y_train, dt_pred_train), rmse(y_train, rf_pred_train)]
test_rmse = [rmse(y_test, lin_pred_test), rmse(y_test, dt_pred_test), rmse(y_test, rf_pred_test)]

x = np.arange(len(names))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - w / 2, train_rmse, w, label="Training RMSE", color="#8fc7d4")
ax.bar(x + w / 2, test_rmse, w, label="Test RMSE", color="#125f76")
ax.set_xticks(x, names)
ax.set_ylabel(f"RMSE ({TARGET_UNIT})")
ax.set_title("Training vs test error (mind the gap)")
ax.legend()
fig.tight_layout()
plt.show()

## 13. Interpret it like an engineer — trends, plausibility, and staying in range

Good test scores are not the whole story. Before trusting a model we check that it behaves the way
the physics says it should, that its predictions are physically possible, and that we are using it
**inside** the range of data it learned from.

**(a) Does the model reproduce the expected trends?** We hold every input at its median and vary one
input at a time from a low value to a high value, then check which way the prediction moves.

In [ ]:
medians = df[FEATURES].median()

def trend_effect(feature, model):
    'Predict at the 10th and 90th percentile of one feature, others at median.'
    low = df[feature].quantile(0.10)
    high = df[feature].quantile(0.90)
    row_low = medians.copy(); row_low[feature] = low
    row_high = medians.copy(); row_high[feature] = high
    p = model.predict(np.array([row_low[FEATURES].to_numpy(),
                                row_high[FEATURES].to_numpy()]))
    return p[1] - p[0]  # change in Δp as the feature increases

expected = {
    "flow_velocity_m_s": "increase",
    "pipe_length_m": "increase",
    "pipe_diameter_m": "decrease",
}
print("Trend checks for the Random Forest (holding other inputs at their median):\n")
for feat, want in expected.items():
    delta = trend_effect(feat, forest)
    got = "increase" if delta > 0 else ("decrease" if delta < 0 else "no change")
    ok = "OK  " if got == want else "WARN"
    print(f"  [{ok}] as {feat:20s} rises, Δp {got:8s} by ~{abs(delta):6.2f} {TARGET_UNIT} "
          f"(expected {want})")

**(b) Are the predictions physically plausible?** Pressure drop cannot be negative.

In [ ]:
neg = int((rf_pred_test < 0).sum())
if neg == 0:
    print(f"None of the {len(rf_pred_test)} test predictions are negative — physically sensible.")
else:
    print(f"WARNING: {neg} of {len(rf_pred_test)} test predictions are negative "
          f"(not physically possible).")

**(c) Are we using the model inside its demonstrated range?** A model learns from the range of data
it was shown. Using it **outside** that range — different fluids, geometries, laminar flow, fittings
and bends — is **extrapolation** and can be unreliable. The check below flags an input row that
falls outside the training ranges.

In [ ]:
train_df = pd.DataFrame(X_train, columns=FEATURES)
lows, highs = train_df.min(), train_df.max()

print("Demonstrated (training) input ranges:")
for f in FEATURES:
    print(f"  {f:22s}: {lows[f]:.4g} .. {highs[f]:.4g}")

def in_domain(row):
    'True if every input is within the training range (interpolation, not extrapolation).'
    return all(lows[f] <= row[f] <= highs[f] for f in FEATURES)

# A deliberately out-of-range example: a very large diameter, far above the training data.
example = {"pipe_length_m": 15.0, "pipe_diameter_m": 0.80, "flow_velocity_m_s": 2.0,
           "fluid_density_kg_m3": 1000.0, "dynamic_viscosity_pa_s": 1.0e-3}
pred = forest.predict(np.array([[example[f] for f in FEATURES]]))[0]
print(f"\nExample prediction: {pred:.2f} {TARGET_UNIT}")
print("In demonstrated range?", in_domain(example),
      "-> treat out-of-range predictions with great caution (extrapolation).")

> **Engineering takeaway.** The model reproduces the expected physical trends and gives plausible,
> in-range predictions on this synthetic data — a good *demonstration*. It still does **not** prove
> the physics, and it must not be used outside the demonstrated range or for real design. Machine
> learning supports engineering judgement; it does not replace it.

## 14. Your turn — short exercises

Try these by editing the cells above (or add new cells here). They take a few minutes each and
build real intuition:

1. **Change the seed.** Set `SEED = 7` at the top and re-run. Do the metrics shift a little? Does
   the *ranking* of the models change? (It usually should not.)
2. **Drop an input.** Remove `"dynamic_viscosity_pa_s"` from `FEATURES` and re-run. Which model is
   most affected?
3. **Change the forest size.** Try `RandomForestRegressor(n_estimators=20, random_state=SEED)` and
   `n_estimators=300`. Does more trees always help the test score?
4. **Make the tree overfit harder.** Set the Decision Tree to `max_depth=None`. What happens to the
   training vs test gap?
5. **Predict your own row.** Build a new input row (inside the demonstrated ranges) and predict its
   pressure drop. Does the number look sensible given the trends?

In [ ]:
# Scratch space for the exercises — edit and run freely.
# Example for exercise 5 (an in-range row):
my_row = {
    "pipe_length_m": 10.0,
    "pipe_diameter_m": 0.10,
    "flow_velocity_m_s": 2.0,
    "fluid_density_kg_m3": 1000.0,
    "dynamic_viscosity_pa_s": 1.0e-3,
}
my_pred = forest.predict(np.array([[my_row[f] for f in FEATURES]]))[0]
print(f"Predicted pressure drop: {my_pred:.2f} {TARGET_UNIT}")
print("In demonstrated range?", in_domain(my_row))

## 15. Where to go next (optional, advanced)

These are pointers, not required steps. Each is a natural next rung once the basics above feel
comfortable:

- **Feature importance** — ask the forest which inputs mattered most (run the cell below).
- **Cross-validation** — instead of a single split, average performance over several splits with
  `sklearn.model_selection.cross_val_score` for a more robust estimate.
- **Gradient boosting** — try `sklearn.ensemble.GradientBoostingRegressor` or
  `HistGradientBoostingRegressor` as an alternative flexible model.
- **Prediction intervals** — quantify uncertainty (e.g. quantile regression, or the spread across
  forest trees) rather than a single number.
- **Your own (non-critical) data** — load a small CSV of your own with the same idea, always keeping
  the synthetic-vs-real and in-domain-vs-out-of-domain cautions in mind.

Remember the framing throughout: this is a **learning tool** for small, non-critical work — not a
validated design or analysis tool.

In [ ]:
# Feature importance from the Random Forest (higher = more influential in this model).
importances = pd.Series(forest.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Random Forest feature importance:")
for name, val in importances.items():
    print(f"  {name:22s} {val:.3f}")

## 16. Optional advanced extension — a neural network

> **Optional and advanced.** Everything you need from this notebook is already above. This section is
> a *next rung*: it introduces a **neural network** — the same kind of model offered as the
> **advanced** option in the browser Explore mode. It is here to show what the code looks like and to
> make an honest point, **not** because a neural network is the right default for this problem.

A **neural network** (here a *multi-layer perceptron*, `MLPRegressor`) learns by combining **layers
of simple units**. Each unit takes a weighted sum of its inputs, passes it through a non-linear
**activation** function (we use `relu`), and feeds the result to the next layer. Stacking layers lets
the network represent curved, non-linear relationships like pressure drop.

A few things matter a great deal for neural networks — and are why they need more care than the
models above:

- **Scaling is essential.** Neural networks train badly on raw, differently-scaled inputs (our pipe
  diameter is ~0.1 while density is ~1000). We put a `StandardScaler` **in the same pipeline**, so the
  scaler is fitted on the training data only and applied consistently — exactly as Explore scales
  inputs for you in the browser.
- **Architecture (layers and neurons).** `hidden_layer_sizes=(32, 16)` means two hidden layers, of 32
  then 16 neurons — the same as Explore's **Medium network** preset. More/larger layers add capacity
  but also cost and overfitting risk.
- **Epochs and learning rate.** Training is **iterative**: it passes over the data many times
  (`max_iter`), each time nudging the weights by a step governed by the **learning rate**
  (`learning_rate_init`). Too large a rate is unstable; too small trains slowly.
- **Early stopping and regularisation.** `early_stopping=True` holds back part of the training data
  and stops once the validation score stops improving, which guards against overfitting;
  `alpha` adds a small **L2 penalty** on the weights for the same reason.
- **Reproducibility.** `random_state=42` fixes the weight initialisation and the internal shuffling,
  so re-running gives the same result.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.neural_network import MLPRegressor

# StandardScaler + MLPRegressor in one pipeline, so scaling is learned on the training data only.
# hidden_layer_sizes=(32, 16) mirrors Explore's "Medium network" preset.
nn = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(32, 16),   # two hidden layers: 32 then 16 neurons
        activation="relu",
        solver="adam",
        alpha=1e-3,                     # L2 regularisation (weight penalty)
        learning_rate_init=0.01,
        max_iter=2000,                  # generous epoch cap; early stopping usually ends sooner
        early_stopping=True,            # hold out 10% of training data to watch for overfitting
        n_iter_no_change=25,            # patience before stopping
        random_state=SEED,
    ),
)

# A neural network may warn if it hits max_iter without fully converging. We do NOT hide that:
# we capture it and report it honestly, because it tells you something about the training.
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", ConvergenceWarning)
    nn.fit(X_train, y_train)

mlp = nn.named_steps["mlpregressor"]
nn_pred_train = nn.predict(X_train)
nn_pred_test = nn.predict(X_test)

converged = not any(issubclass(w.category, ConvergenceWarning) for w in caught)
print("Neural network (MLPRegressor, hidden layers = (32, 16))")
print(f"  Train : R2 = {r2_score(y_train, nn_pred_train):.3f}   "
      f"RMSE = {rmse(y_train, nn_pred_train):6.2f} {TARGET_UNIT}")
print(f"  Test  : R2 = {r2_score(y_test, nn_pred_test):.3f}   "
      f"RMSE = {rmse(y_test, nn_pred_test):6.2f} {TARGET_UNIT}")
print(f"  Trained for {mlp.n_iter_} iterations "
      f"({'converged / stopped early' if converged else 'hit max_iter — see note below'}).")

### The training (loss) curve

Because a neural network trains **iteratively**, we can watch its **loss** fall epoch by epoch.
`MLPRegressor` records this in `loss_curve_`. A curve that falls and then flattens has effectively
converged; if it were still dropping steeply at the end, a larger `max_iter` might help. This is a
diagnostic — the numbers that matter for the engineering question are still the **test** RMSE and R².

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# (left) training loss per epoch
axes[0].plot(range(1, len(mlp.loss_curve_) + 1), mlp.loss_curve_, color="#125f76")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Training loss")
axes[0].set_title("Neural network training curve")

# (right) actual vs predicted on the test set
lo = float(min(y_test.min(), nn_pred_test.min()))
hi = float(max(y_test.max(), nn_pred_test.max()))
axes[1].scatter(y_test, nn_pred_test, s=28, alpha=0.7, color="#125f76", label="Test rows")
axes[1].plot([lo, hi], [lo, hi], "--", color="#a52b2b", label="Perfect prediction")
axes[1].set_xlabel(f"Actual pressure drop ({TARGET_UNIT})")
axes[1].set_ylabel(f"Predicted pressure drop ({TARGET_UNIT})")
axes[1].set_title("Neural network: actual vs predicted (test)")
axes[1].legend()

fig.tight_layout()
plt.show()

# Add the neural network to the comparison table so it sits beside the other models.
results_with_nn = pd.concat([
    results,
    pd.DataFrame([scores("Neural Network", nn_pred_train, nn_pred_test)]),
], ignore_index=True)
results_with_nn

### Why the score above is lower than the browser's — training length

The number in the table above (test R² ≈ 0.79) is noticeably lower than the browser Explore mode,
where the neural network reaches ≈ 0.99. It is tempting to conclude "the browser network is better"
or "a neural network can't beat the Random Forest here" — but **both readings are wrong**. The gap is
an **early-stopping artifact**, not a real difference in model quality.

`early_stopping=True` with `n_iter_no_change=25` watches a small internal validation slice and, on
this easy data, halts after only about **50 iterations** — before the network has finished learning.
The quick check below refits the **same architecture** with early stopping switched off and lets it
train to convergence. It reaches a much higher test R², close to the browser's, confirming the low
score above is an **underfit** rather than a limit of the model.


In [ ]:
# Diagnostic: the score above depends heavily on WHEN training stops.
# With early_stopping=True the network above halted after only ~50 iterations (an underfit).
# Trained to convergence (early stopping off), the SAME architecture fits much better.
# This is a teaching point about training length, NOT a recommended change to the default.
nn_converged = make_pipeline(
    StandardScaler(),
    MLPRegressor(
        hidden_layer_sizes=(32, 16), activation="relu", solver="adam",
        alpha=1e-3, learning_rate_init=0.01, max_iter=2000,
        early_stopping=False, random_state=SEED,
    ),
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    nn_converged.fit(X_train, y_train)
conv_pred_test = nn_converged.predict(X_test)
conv_iters = nn_converged.named_steps["mlpregressor"].n_iter_
print("Same architecture, trained to convergence (early_stopping=False):")
print(f"  iterations   = {conv_iters}")
print(f"  Test R2      = {r2_score(y_test, conv_pred_test):.3f}   "
      f"(early-stopping run above = {r2_score(y_test, nn_pred_test):.3f})")
print("  => the lower score above is an early-stopping underfit, not a limit of the model;")
print("     trained to convergence this network approaches the browser's ~0.99 on this easy data.")


> **Honest takeaway.** Read the two neural-network numbers together. Trained to convergence, this
> network fits the data very well (test R² ≈ 0.99) — but that is because the dataset is **smooth and
> low-noise** (its irreducible-noise R² ceiling is about 0.996), *not* because a neural network is
> the right tool for engineering problems in general. On **real, noisier, smaller** tabular datasets,
> well-tuned tree-based methods such as the Random Forest are frequently as good as — or better than
> — a neural network, and they need far less care (scaling, more settings, training length,
> reproducibility). "More complex" is **not** the same as "more accurate" or "more suitable for
> engineering use". The early-stopping run is also a concrete reminder that **training length
> matters**: stop too soon and even a capable model underfits.
>
> The same cautions as every other model still apply: this is synthetic data, the network must not be
> used outside the demonstrated range, and physical validation and engineering judgement remain
> essential. This mirrors the browser Explore mode, which is why the neural network is offered there
> as an **advanced** option rather than the beginner default.

---

*Engineering ML Studio is a redevelopment of **Local Regression Studio** by **Yu Duan** (MIT
licence). New contributions in this project are © UKRI, under the same MIT terms, and do not alter
the original work's copyright or attribution. This notebook and its dataset are a **synthetic
teaching demonstration** — not validated engineering data.*